In [40]:
import numpy as np
import pandas as pd
from skimage import measure
from pathlib import Path
from ngio import open_ome_zarr_container
from ngio.utils import download_ome_zarr_dataset

from skimage import measure, feature, filters
from ngio.transforms import ZoomTransform
import napari
import tqdm
import ngff_zarr as nz

In [2]:
viewer = napari.Viewer()

In [ ]:
# Download a sample dataset
download_dir = Path("./data")
hcs_path = download_dir / "20200812-CardiomyocyteDifferentiation14-Cycle1_mip.zarr"
image_path = hcs_path / "B" / "03" / "0"

ome_zarr = open_ome_zarr_container(image_path)

In [47]:
def find_maxima(image: np.ndarray, min_distance: int = 5) -> pd.DataFrame:
    """Find local maxima in the image using skimage.feature.peak_local_max."""
    image_blurred = filters.gaussian(image.squeeze(), sigma=15)
    coordinates = feature.peak_local_max(image_blurred, min_distance=min_distance)
    return pd.DataFrame(coordinates, columns=["y", "x"])

def extract_features(label: np.ndarray) -> pd.DataFrame:
    """Basic feature extraction using skimage.measure.regionprops_table."""
    label = label.squeeze(-1)  # Remove the channel axis if present
    roi_feat_table = measure.regionprops_table(
        label_image=label,
        properties=[
            "label",
            "area",
            "bbox",
        ],
    )
    return pd.DataFrame(roi_feat_table)

In [48]:
# First we will need the image object and the FOVs table
image = ome_zarr.get_image()

# Get the nuclei label
nuclei = ome_zarr.get_label("nuclei")
nuclei.shape

(1, 4320, 5120)

In [49]:
# We need to setup a transform to resample the nuclei to the image resolution
zoom_transform = ZoomTransform(
    input_image=nuclei,
    target_image=image,
    order="nearest",  # Nearest neighbor interpolation for labels
)

In [50]:
from ngio.experimental.iterators import FeatureExtractorIterator
from ngio.tables import FeatureTable


iterator = FeatureExtractorIterator(
    input_image=image,
    input_label=nuclei,
    label_transforms=[zoom_transform],
    axes_order=["y", "x", "c"],
)

In [53]:
maxima_feat_table = []
roi_feature_table = []
ch_idx = image.get_channel_idx("DAPI")
for image_data, label_data, roi in tqdm.tqdm(iterator.iter_as_numpy()):
    print(f"Processing ROI: {roi}")
    maxima_feat_table.append(find_maxima(image=image_data[:, :, ch_idx]))
    roi_feature_table.append(extract_features(label=label_data))

0it [00:00, ?it/s]

Processing ROI: Roi(t=None, z=0.0->1.0, y=0.0->702.0, x=0.0->832.0)


1it [00:05,  5.18s/it]


In [54]:
maxima_feat_table = pd.concat(maxima_feat_table, ignore_index=True)
roi_feature_table = pd.concat(roi_feature_table, ignore_index=True)

In [57]:
left = roi_feature_table['bbox-0']
top = roi_feature_table['bbox-1']
right = roi_feature_table['bbox-2']
bottom = roi_feature_table['bbox-3']

boxes = []
for i in range(len(left)):
    box_corners = np.array([
        [left[i], top[i]],
        [right[i], top[i]],
        [right[i], bottom[i]],
        [left[i], bottom[i]]
    ])
    boxes.append(box_corners)


In [44]:
ngff_image = nz.from_ngff_zarr(image_path)
ms = [scale.data[0] for scale in ngff_image.images]

In [58]:
viewer.layers.clear()
viewer.add_image(ms)
viewer.add_points(maxima_feat_table[["y", "x"]].values, size=50, face_color="red")
viewer.add_shapes(boxes, shape_type='polygon', edge_color='blue', face_color='transparent')

<Shapes layer 'boxes' at 0x2c4298201a0>